In [ ]:
### Preprocessing

import sys, os
sys.path.insert(0, os.path.expanduser("~/Sleep_Stage_Research/conference"))

import config as cfg
from data_utils import preprocess_all_subjects, load_participant_info, get_subject_list
import numpy as np

print(f"Dataset root: {cfg.DATASET_ROOT}")
print(f"Preprocessed dir: {cfg.PREPROCESSED_DIR}")
print(f"Channels: {cfg.PSG_CHANNELS}")
print(f"Epoch: {cfg.EPOCH_SEC}s = {cfg.EPOCH_SAMPLES} samples @ {cfg.SAMPLING_RATE}Hz")

h5_path = preprocess_all_subjects()
print(f"\nHDF5 file: {h5_path}")
print(f"Size: {os.path.getsize(h5_path) / (1024**3):.2f} GB")

In [ ]:
### Verify HDF5 and show per-subject stats

import h5py
import pandas as pd

h5_path = os.path.join(cfg.PREPROCESSED_DIR, "dreamt_psg7ch_epochs.h5")
subjects = get_subject_list()

rows = []
with h5py.File(h5_path, "r") as hf:
    for subj in sorted(hf.keys()):
        labels = hf[f"{subj}/labels"][:]
        epochs_shape = hf[f"{subj}/epochs"].shape
        counts = {cfg.STAGE_NAMES[i]: int(np.sum(labels == i)) for i in range(cfg.NUM_CLASSES)}
        counts["total"] = len(labels)
        counts["hours"] = len(labels) * 30 / 3600
        counts["subject"] = subj
        counts["shape"] = str(epochs_shape)
        rows.append(counts)

df_stats = pd.DataFrame(rows)
cols_order = ["subject", "shape", "W", "N1", "N2", "N3", "R", "total", "hours"]
print(df_stats[cols_order].to_string(index=False))

print(f"\nTotal subjects: {len(df_stats)}")
print(f"Total epochs: {df_stats['total'].sum():,}")
print(f"Total hours: {df_stats['hours'].sum():.1f}")

print(f"\nGlobal class distribution:")
for s in cfg.STAGE_NAMES:
    total = df_stats[s].sum()
    pct = 100 * total / df_stats["total"].sum()
    print(f"  {s}: {total:,} ({pct:.1f}%)")

In [ ]:
### Step 3 — Load participant info and define CV folds

from sklearn.model_selection import KFold

pinfo = load_participant_info()
subjects = sorted(get_subject_list())

print(f"Subjects in HDF5: {len(subjects)}")
print(f"Subjects in participant_info: {len(pinfo)}")

kf = KFold(n_splits=cfg.NUM_FOLDS, shuffle=True, random_state=cfg.SEED)
folds = {}
for fold_idx, (train_idx, test_idx) in enumerate(kf.split(subjects)):
    train_subjects = [subjects[i] for i in train_idx]
    test_subjects = [subjects[i] for i in test_idx]
    folds[fold_idx] = {"train": train_subjects, "test": test_subjects}
    print(f"Fold {fold_idx}: train={len(train_subjects)}, test={len(test_subjects)} -> {test_subjects}")

# Save fold assignments
import json
folds_path = os.path.join(cfg.CHECKPOINT_DIR, "fold_assignments.json")
with open(folds_path, "w") as f:
    json.dump(folds, f, indent=2)
print(f"\nFold assignments saved to {folds_path}")

In [ ]:
### Step 4 — Verify signal ranges 

import matplotlib.pyplot as plt

h5_path = os.path.join(cfg.PREPROCESSED_DIR, "dreamt_psg7ch_epochs.h5")
subj = subjects[0]

with h5py.File(h5_path, "r") as hf:
    epochs = hf[f"{subj}/epochs"][:5]  # first 5 epochs
    labels = hf[f"{subj}/labels"][:5]

fig, axes = plt.subplots(len(cfg.PSG_CHANNELS), 1, figsize=(14, 2.5 * len(cfg.PSG_CHANNELS)), sharex=True)
for ch_idx, ch_name in enumerate(cfg.PSG_CHANNELS):
    signal = epochs[:, :, ch_idx].flatten()
    axes[ch_idx].plot(signal, linewidth=0.3)
    axes[ch_idx].set_ylabel(ch_name, fontsize=9)
    axes[ch_idx].tick_params(labelsize=7)

epoch_boundaries = np.arange(0, len(signal), cfg.EPOCH_SAMPLES)
for eb in epoch_boundaries:
    axes[0].axvline(x=eb, color="red", linestyle="--", alpha=0.5, linewidth=0.5)

axes[0].set_title(f"Subject {subj} - First 5 epochs (labels: {[cfg.STAGE_NAMES[l] for l in labels]})")
axes[-1].set_xlabel("Sample")
plt.tight_layout()
plt.show()

print(f"Signal ranges per channel (subject {subj}, first 5 epochs):")
for ch_idx, ch_name in enumerate(cfg.PSG_CHANNELS):
    vals = epochs[:, :, ch_idx].flatten()
    print(f"  {ch_name}: min={vals.min():.6f}, max={vals.max():.6f}, mean={vals.mean():.6f}, std={vals.std():.6f}")